In [1]:
### ============ 1. IMPORTS =====================
import pandas as pd # for data manipulation
import pickle # for saving the processed data to disk
import sys # for manipulating the Python path
import os # for file path manipulations
from scipy.io import arff # for loading ARFF files from OpenML
# !pip install ucimlrepo # for loading datasets from the UCI Machine Learning Repository
# from ucimlrepo import fetch_dataset # for fetching datasets from the UCI Machine Learning Repository

# Make the src/ package importable from within the notebooks/ subfolder.
sys.path.append("..")

# Pull in the same SEED and TEST_SIZE constants from config.py so that all notebooks use the same values.
from src.config import SEED, TEST_SIZE
from src.preprocess import split_data, scale_data

In [2]:
# # ===================== DATASET 1 - UCI Phishing Website Dataset (Full Variant) [16] =====
# # Load the data file
# data, meta = arff.loadarff('../data/raw/Phishing_Legitimate_full.arff')
# # Convert to a Pandas data frame
# df = pd.DataFrame(data)
# # inspect the data
# df.head()

In [3]:
# # DATASET 3 - UCI Phishing Website Dataset (Small Variant) [18]
# # load the data file
# df = pd.read_csv('../data/raw/dataset_small.csv')
# # inspect the data
# print(df.shape)
# df.head()

In [4]:
# DATASET 4 - UCI Phishing Website Dataset (Full Variant) [19]
# load the data file
df = pd.read_csv('../data/raw/dataset_full.csv')
# inspect the data
print(df.shape)
df.head()

(88647, 112)


,qty_dot_url,qty_hyphen_url,qty_underline_url,qty_slash_url,qty_questionmark_url,qty_equal_url,qty_at_url,qty_and_url,qty_exclamation_url,qty_space_url,...,qty_ip_resolved,qty_nameservers,qty_mx_servers,ttl_hostname,tls_ssl_certificate,qty_redirects,url_google_index,domain_google_index,url_shortened,phishing
0,3,0,0,1,0,0,0,0,0,0,...,1,2,0,892,0,0,0,0,0,1
1,5,0,1,3,0,3,0,2,0,0,...,1,2,1,9540,1,0,0,0,0,1
2,2,0,0,1,0,0,0,0,0,0,...,1,2,3,589,1,0,0,0,0,0
3,4,0,2,5,0,0,0,0,0,0,...,1,2,0,292,1,0,0,0,0,1
4,2,0,0,0,0,0,0,0,0,0,...,1,2,1,3597,0,1,0,0,0,0


In [5]:
# ===== 3. DATA INSPECTION =====
# check the shape to confirm we have the expected number of rows and columns
# print(df.shape)
# # show summary statistics and data types to check for any obvious issues
# print(df.info()) 
# # check for missing values and duplicates
# print(df.isnull().sum())
# #  check if there are duplicates so they can be removed before training the GA
# print(df.duplicated().sum())

In [6]:
# ===== 4. PREPROCESSING =====
# Identify target and features
target_col = "phishing" # the name of the target variable column in the dataset

# Drop the label column to get the feature matrix X.
X = df.drop(columns=[target_col])

# convert bytes to int
y = df[target_col].astype(str).astype(int)

print("X shape:", X.shape[1])   # number of features
print("y shape:", y.shape)      # number of samples

# check the distribution of the target variable
print(y.value_counts())
print(y.unique())

X shape: 111
y shape: (88647,)
phishing
0    58000
1    30647
Name: count, dtype: int64
[1 0]


In [7]:
# DATASET 2 - UCI Phishing Website Dataset [17]
# # load the data file
# data, meta = arff.loadarff('../data/raw/.old.arff')

# # Convert to a Pandas data frame
# df = pd.DataFrame(data)

# # convert bytes to int
# for col in df.select_dtypes(include=["object"]).columns:
#     df[col] = df[col].apply(lambda x: x.decode("utf-8") if isinstance(x, bytes) else x)

# # convert to int
# df = df.astype(int)

# # fix the target variable to be binary 0/1 instead of -1/1
# df["Result"] = df["Result"].replace({-1: 0, 1: 1})

# # inspect the data
# df.head()

# # split the data into features and target
# X = df.drop(columns=["Result"])
# y = df["Result"]

In [8]:
# ===== 6. TRAIN / TEST SPLIT =====
# Use the same SEED and TEST_SIZE constants from config.py to ensure all notebooks use the same split.
X_train, X_test, y_train, y_test = split_data(X, y, test_size=TEST_SIZE, seed=SEED)
print(X_train.shape, X_test.shape)  # confirm 70/30 ratio looks right

(62052, 111) (26595, 111)


In [9]:
# ===== 7. SCALE FEATURES =====
# scale the data using z-score normalization (standardization)
X_train_scaled, X_test_scaled, mean, std_replaced = scale_data(X_train, X_test)

# Verify the scaling worked by checking the mean and standard deviation of the scaled features.
print(X_train_scaled.describe().loc[["mean", "std"]])

       qty_dot_url  qty_hyphen_url  qty_underline_url  qty_slash_url  \
mean  1.607687e-16    2.244350e-17      -7.099473e-18  -2.198546e-17   
std   1.000000e+00    1.000000e+00       1.000000e+00   1.000000e+00   

      qty_questionmark_url  qty_equal_url    qty_at_url   qty_and_url  \
mean          1.099273e-17   2.290153e-18 -2.244350e-17 -2.335956e-17   
std           1.000000e+00   1.000000e+00  1.000000e+00  1.000000e+00   

      qty_exclamation_url  qty_space_url  ...  time_domain_expiration  \
mean         7.786519e-18  -9.160610e-19  ...           -1.402718e-17   
std          1.000000e+00   1.000000e+00  ...            1.000000e+00   

      qty_ip_resolved  qty_nameservers  qty_mx_servers  ttl_hostname  \
mean     1.250423e-16    -1.601962e-16    8.152943e-17 -1.305387e-17   
std      1.000000e+00     1.000000e+00    1.000000e+00  1.000000e+00   

      tls_ssl_certificate  qty_redirects  url_google_index  \
mean        -9.893459e-17  -1.282485e-17      2.015334e-17   
st

In [10]:
# ===== 8. SAVE PROCESSED DATA =====
# Persist the split and scaled arrays to disk with pickle so that
# subsequent notebooks (02, 03, 04) can reload them without re-running
# the entire preprocessing pipeline from scratch.
with open("../data/processed/uci_full/uci_full_split_scaled.pkl", "wb") as f:
    pickle.dump((X_train_scaled, X_test_scaled, y_train, y_test), f)